In [1]:
import numpy as np
import pandas as pd
import glob
import os

# ================= 1. 数据加载与预处理 (只需运行一次) =================
# 请确保这里的路径与你保存数据时的路径一致
base_path = r'D:\Workspace\5_Data\bupt_ring_selftest\wsx\报告\green\parsed_results'

# 1. 加载并拼接原始数据 (保证与 ac_signal 长度绝对对齐)
search_pattern = os.path.join(base_path, '202*.csv')
file_list = sorted(glob.glob(search_pattern))

if not file_list:
    print("❌ 未找到CSV文件，请检查 base_path 路径！")
else:
    print("⏳ 正在读取并拼接原始 CSV 数据，请稍候...")
    df_list = [pd.read_csv(f) for f in file_list]
    df = pd.concat(df_list, ignore_index=True)
    
    # 极性反转设置
    INVERT_POLARITY = True
    polarity_multiplier = -1.0 if INVERT_POLARITY else 1.0
    
    # 填充缺失值并应用极性反转
    raw_signal = df['green1'].ffill().fillna(0).values * polarity_multiplier
    print(f"✅ 原始数据加载完成，总点数: {len(raw_signal)}")

# 2. 加载 One-pass 重构的 AC 信号
npy_path = os.path.join(base_path, 'ac_signal_onepass.npy')
if not os.path.exists(npy_path):
    print(f"❌ 未找到 {npy_path}，请确保前面的流式处理脚本已执行成功！")
else:
    ac_signal = np.load(npy_path)
    print(f"✅ 滤波数据加载完成，总点数: {len(ac_signal)}")

⏳ 正在读取并拼接原始 CSV 数据，请稍候...
✅ 原始数据加载完成，总点数: 8262961
✅ 滤波数据加载完成，总点数: 8262961


In [2]:
# 1. 声明模式
%matplotlib inline 

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display
import numpy as np

# ================= 2. 交互式可视化模块 =================
if 'raw_signal' in locals() and 'ac_signal' in locals():
    total_points = len(ac_signal)
    fs = 100.0 
    
    # 【防御性编程】暴力清理由于之前中断残留的隐形图表，释放内存
    plt.close('all') 

    def interactive_onepass_plot(start_idx, window_size):
        # 避免前端 DOM 堆叠卡死
        clear_output(wait=True) 
        
        end_idx = min(start_idx + window_size, total_points)
        time_axis = np.arange(start_idx, end_idx) / fs
        
        raw_slice = raw_signal[start_idx:end_idx]
        ac_slice = ac_signal[start_idx:end_idx]
        
        # 显式声明图表并设置内存回收机制
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
        
        # -------- 子图 1: 原始 PPG --------
        ax1.plot(time_axis, raw_slice, color='gray', linewidth=1.5, label='Raw PPG Signal')
        ax1.set_title(f'One-pass Nonparametric Regression (Time: {start_idx/fs:.1f}s to {end_idx/fs:.1f}s)', fontsize=14, pad=10)
        ax1.set_ylabel('Raw Amplitude', fontsize=12)
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        
        # -------- 子图 2: One-pass 重构交流波形 --------
        local_raw_centered = raw_slice - np.mean(raw_slice)
        # ax2.plot(time_axis, local_raw_centered, color='lightgray', alpha=0.6, label='Raw (Local Centered)')
        ax2.plot(time_axis, ac_slice, color='red', linewidth=2.5, label='One-pass Reconstructed AC')
        
        ax2.set_xlabel('Time (Seconds)', fontsize=12)
        ax2.set_ylabel('AC Amplitude', fontsize=12)
        ax2.axhline(0, color='black', linestyle='--', linewidth=0.8) 
        ax2.legend(loc='upper right')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # 【关键】绘图结束后立刻、彻底销毁当前 fig 对象
        plt.close(fig) 

    # ================= 核心修复：显式构建 UI 架构 =================
    max_start = max(0, total_points - 500) 

    # 1. 显式实例化滑条对象
    slider_start = widgets.IntSlider(
        min=0, max=max_start, step=200, value=0, 
        description='起点索引:', continuous_update=False, 
        layout=widgets.Layout(width='400px')
    )
    
    slider_window = widgets.IntSlider(
        min=200, max=5000, step=100, value=1000, 
        description='窗口大小:', continuous_update=False,
        layout=widgets.Layout(width='400px')
    )
    
    # 2. 将滑条与绘图函数绑定，生成专门的输出区域
    out = widgets.interactive_output(
        interactive_onepass_plot, 
        {'start_idx': slider_start, 'window_size': slider_window}
    )
    
    # 3. 将两个滑条横向排列
    ui = widgets.HBox([slider_start, slider_window])
    
    # 4. 显式渲染界面 (上面是控制区，下面是绘图区)
    display(ui, out)

else:
    print("⚠️ 未找到 raw_signal 或 ac_signal，请先运行数据加载的 Cell！")

Output()